!python -m pip install --upgrade pip
!pip install scanpy

In [1]:
import scanpy as sc
import pandas as pd
from recon.explore import Celltype
import numpy as np
import scanpy as sc  # single cell data
import pandas as pd  # data manipulation
import liana as li  # cell communication
import recon  # multilayer and perturbation prediction
import recon.data

In [2]:
path = "10M_PBMC_12donor_90cytokines_h5ad_20260203/20260203_Parse_10M_PBMC_cytokines.h5ad"
rna = sc.read_h5ad(path) # backed = "r"
print (rna.shape)

(9697974, 40352)


In [3]:
print (rna.obs["treatment"].unique())


['cytokine', 'PBS']
Categories (2, object): ['PBS', 'cytokine']

In [4]:
rna = rna[rna.obs["treatment"] == "PBS"]

In [5]:
print (rna)

View of AnnData object with n_obs × n_vars = 629701 × 40352
    obs: 'sample', 'species', 'gene_count', 'tscp_count', 'mread_count', 'bc1_wind', 'bc2_wind', 'bc3_wind', 'bc1_well', 'bc2_well', 'bc3_well', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'total_counts_MT', 'pct_counts_MT', 'log1p_total_counts_MT', 'donor', 'cytokine', 'treatment', 'cell_type'
    var: 'n_cells'


### Loading the GRNs

In [8]:
grn_path = "./GRMs_by_Pau_to_share/pbmc_hummus.csv"
grn = pd.read_csv(grn_path)
grn = grn.rename(columns={"score": "weight"})

In [9]:
grn

,source,cre,target,weight
0,TFAP2C,chr5-40410239-40410739,RPL37,0.000163
1,TFAP2C,chr5-40439075-40439575,RPL37,0.000163
2,TFAP2C,chr5-40486302-40486802,RPL37,0.000163
3,TFAP2C,chr5-40486873-40487373,RPL37,0.000163
4,TFAP2C,chr5-40502665-40503165,RPL37,0.000163
...,...,...,...,...
99998,VEZF1,chr20-59004826-59005326,GNAS,0.000022
99999,VEZF1,chr20-59016634-59017134,GNAS,0.000022
100000,VEZF1,chr20-59120435-59120935,GNAS,0.000022
100001,VEZF1,chr20-59161726-59162226,GNAS,0.000022


### Cell-Cell Communication

In [12]:
li.method.cellphonedb(rna, 
            # NOTE by default the resource uses HUMAN gene symbols
            resource_name="consensus", # mouseconsensus
            expr_prop=0.00,
            use_raw=False,
            groupby="cell_type",
            verbose=True, key_added='cpdb_res')
            

Using resource `consensus`.
Using `.X`!
/homes/shree/miniforge3/envs/recon/lib/python3.10/site-packages/anndata/_core/anndata.py:430: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
Make sure that normalized counts are passed!
/homes/shree/miniforge3/envs/recon/lib/python3.10/site-packages/liana/method/_pipe_utils/_pre.py:168: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
['Y_RNA-18', 'Y_RNA-19', 'Y_RNA-21', 'Y_RNA-29', 'Y_RNA-33', 'Y_RNA-35', 'Y_RNA-43', 'Y_RNA-46', 'Y_RNA-62', 'Y_RNA-65', 'Y_RNA-66', 'Y_RNA-67', 'Y_RNA-68', 'Y_RNA-79', 'Y_RNA-85', 'Y_RNA-86', 'Y_RNA-93', 'Y_RNA-101', 'Y_RNA-121', 'Y_RNA-126', 'Y_RNA-127', 'Y_RNA-137', 'Y_RNA-163', 'Y_RNA-164', 'Y_RNA-176', 'Y_RNA-191', 'Y_RNA-196', 'Y_RNA-198', 'Y_RNA-215', 'Y_RNA-240', 'Y_RNA-247', 'Y_RNA-248', '

Generating ligand-receptor stats for 629701 samples and 1766 features


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [04:45<00:00,  3.50it/s]
/homes/shree/miniforge3/envs/recon/lib/python3.10/site-packages/liana/method/sc/_Method.py:319: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.


In [13]:
ccc_network = rna.uns["cpdb_res"].copy()
ccc_network = ccc_network[["ligand", "receptor", "lr_means", "source", "target"]]
ccc_network = ccc_network.rename(columns={
    "lr_means": "weight",
    "source": "celltype_source",
    "target": "celltype_target",
    "ligand": "source",
    "receptor": "target"
})
ccc_network = ccc_network[ccc_network['weight'] != 0]


### Save ccc_network to a file to save time

In [14]:
ccc_network.to_csv("ccc_network.csv", index = False)

### Load Receptor Gene links

In [15]:
receptor_genes = recon.data.load_receptor_genes("human_receptor_gene_from_NichenetPKN")
# for human, use "human_receptor_gene_from_NichenetPKN"

genes = np.unique(grn['source'].tolist() + grn['target'].tolist())
receptor_genes = receptor_genes[receptor_genes['target'].isin(genes)]
receptor_genes.head()

,source,target,weight
2,A1BG,ABCB1,0.006146
4,A1BG,ACKR3,0.005384
5,A1BG,ACOT7,0.005152
7,A1BG,ACSL1,0.006188
9,A1BG,ADAMTS1,0.005072


### Select a molecule to perturb

In [16]:
# variance
ccc_network.groupby("target")["weight"].var().sort_values(ascending=False).head(3)

target
CD1A    18.560102
CD1C    18.538691
CD74    18.147257
Name: weight, dtype: float32

In [17]:
ligands = ccc_network[ccc_network["target"]=="CD74"]['source'].unique().tolist()
ligands

['APP', 'COPA', 'MIF']

### Run treatment effective perturbation

In [23]:
%%time

# The seed has to change for any effects.
direct_effect, indirect_effect = recon.explore.multicell_targets(
        seeds=["IL32"], 
        celltypes=['CD4 Memory', 'NK', 'NKT', 'pDC', 'B Intermediate/Memory', 'CD8 Memory'],
        grn=grn,
        receptor_grn=receptor_genes,
        ccc=ccc_network,
        grn_graph_weighted=True,
        receptor_grn_graph_weighted=True,
        receptor_graph_weighted=False,
        cell_communication_graph_weighted=True,
        cell_communication_graph_directed=False,
        restart_proba=0.6,
        extend_seeds=True,
        njobs=15
    )

Processing celltype 1/6: CD4 Memory
Processing celltype 2/6: NK
Processing celltype 3/6: NKT
Processing celltype 4/6: pDC


/homes/shree/miniforge3/envs/recon/lib/python3.10/site-packages/recon/explore/recon.py:132: UserWarning: 
                No receptor_graph provided,
                an empty receptor graph will be created.
                


Processing celltype 5/6: B Intermediate/Memory
Processing celltype 6/6: CD8 Memory


/homes/shree/miniforge3/envs/recon/lib/python3.10/site-packages/recon/explore/recon.py:474: UserWarning: The celltypes dictionary was converted to a list of Celltype objects.
The keys of the dictionary will be the celltype names.


Computing intracellular contributions and direct effect...
Incorrect lamb, the lamb[k,k] term need to equal to sum(lamb[k,:]
the 4 column is incorrect


StopIteration: 

### Save direct and indirect effects both of them

In [24]:
total_effect = recon.explore.combine_effects(direct_effect, indirect_effect, alpha=0.8)
total_effect.head()

NameError: name 'direct_effect' is not defined

total_effect.plot.scatter(x='CD4 Memory', y='NK')

recon.plot.plot_celltype_comparison(total_effect, "CD4 Memory", "NK", quantile=0.998)

### Evaluate the results

In [ ]:
dct_ct = {"CD4 Memory": 'CD4_Memory_T_cell', 
"NK" : "NK",
"pDC" : "pDC",
"B Intermediate/Memory" : "Intermediate_B_cell",
"CD8 Memory" : "CD8_Memory_T_cell"
}

In [25]:
gt = pd.read_csv("human_cytokine_dict_mini.csv")

In [26]:
gt

,Unnamed: 0,index,gene,log_fc,logCPM,F,p_value,adj_p_value,contrast,celltype,...,median_num_cells_pbs,median_num_cells_cytokine,mean_num_cells_pbs,mean_num_cells_cytokine,min_num_cells_pbs,min_num_cells_cytokine,max_num_cells_pbs,max_num_cells_cytokine,num_DE_pbs_wells,well_biased
0,0,182,AACS,0.506575,4.404342,19.205548,3.586696e-04,5.992502e-03,NaN,Intermediate_B_cell,...,1435.5,162.0,1814.333333,201.666667,367.0,43.0,4609.0,414.0,4.0,False
1,1,415,AAMDC,-0.592411,4.404723,28.952278,5.144025e-05,7.192403e-04,NaN,Intermediate_B_cell,...,1435.5,181.0,1814.333333,282.916667,367.0,113.0,4609.0,838.0,4.0,False
2,2,423,AAMDC,-0.574430,4.394900,29.933840,4.591638e-05,5.666800e-04,NaN,Intermediate_B_cell,...,1435.5,305.5,1814.333333,382.916667,367.0,120.0,4609.0,1296.0,4.0,False
3,3,437,AAMDC,1.123624,5.038960,157.966713,7.296158e-11,1.083115e-07,NaN,Intermediate_B_cell,...,1435.5,282.0,1814.333333,325.250000,367.0,95.0,4609.0,901.0,6.0,False
4,4,445,AAMDC,0.558964,4.716018,35.557344,2.372101e-06,9.600669e-05,NaN,Intermediate_B_cell,...,1435.5,248.0,1814.333333,311.500000,367.0,83.0,4609.0,655.0,5.0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
371419,371419,16302821,ZYX,-0.541896,6.707809,197.335313,2.154449e-10,6.909837e-08,NaN,Mono,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,False
371420,371420,16302866,ZYX,-0.616012,6.608767,52.620647,5.673225e-06,2.490351e-05,NaN,Mono,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.0,False
371421,371421,16302889,ZYX,-0.512620,6.713729,126.179237,1.254963e-08,1.567677e-06,NaN,Mono,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,False
371422,371422,16302929,ZZEF1,0.613855,8.052541,105.305000,8.553788e-08,6.979058e-06,NaN,Mono,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.0,False


In [ ]:
gt.columns

In [28]:
len(gt["celltype"].unique())

24

In [ ]:
from sklearn.metrics import roc_auc_score

aurocs = {"CD4 Memory": 0, 
"NK" : 0,
"pDC" : 0,
"B Intermediate/Memory" : 0,
"CD8 Memory" : 0
}

for ct in dct_ct.keys():
    truth = gt[gt["celltype"] == dct_ct[ct]].copy()

    print (truth.shape)
    pred = total_effect[ct].rename("score").reset_index()
    pred.columns = ["gene", "score"]

    
    # Binary labels
    truth["truth"] = (
        (truth["adj_p_value"] < 0.05) &
        (truth["log_fc"] > 0)
    ).astype(int)

    truth = truth[["gene", "truth"]]
    #print (truth) 
    # If duplicate genes exist, keep the maximum truth value
    truth = truth.groupby("gene", as_index=False)["truth"].max()
    
    # -------------------------
    # Merge on common genes
    # -------------------------
    
    merged = pred.merge(truth, on="gene", how="inner")
   
    print(f"Predicted genes : {len(pred)}")
    print(f"Truth genes     : {len(truth)}")
    print(f"Common genes    : {len(merged)}")

    auc = roc_auc_score(
        merged["truth"],
        merged["score"]
    )
    
    print(f"AUROC: {auc:.4f}")
    aurocs[ct] = auc


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7,4))

bars = plt.bar(aurocs.keys(), aurocs.values())

plt.ylabel("AUROC", fontsize=12, fontweight="bold")
plt.xlabel("Cell Type", fontsize=12, fontweight="bold")
plt.ylim(0, 1)

# Bold tick labels
plt.xticks(rotation=45, ha="right", fontsize=11, fontweight="bold")
plt.yticks(fontsize=11, fontweight="bold")

# Add AUROC values on top of bars
for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width()/2,
        height + 0.01,
        f"{height:.2f}",
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold"
    )

plt.tight_layout()
plt.show()